# MNIST Case Align Under Controlled Single-Transform Augmentation

This notebook probes whether Case Align degrades as one visually interpretable augmentation becomes stronger.

For selected interesting MNIST cases, it:
- generates `k=10` deterministic steps for each augmentation variant,
- runs separate **rotation**, **shear**, and **scale** tests,
- uses three rows per variant: **low**, **mild**, and **intense**,
- scores the original instance against that row's generated same-class samples,
- checks whether each generated image is still predicted as the original target class,
- generates explanations for retained same-class samples,
- computes cosine Case Align in both problem and solution space.

The output filenames include the augmentation variant so the different tests can be compared side by side.


In [ ]:
from pathlib import Path
import json
import sys
import ssl

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from captum.attr import DeepLift, IntegratedGradients
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

def _configure_ssl_for_macos() -> None:
    ssl._create_default_https_context = ssl._create_unverified_context

cwd = Path.cwd().resolve()
if (cwd / 'mnist_explain_predictions.py').exists():
    SRC_ROOT = cwd
elif (cwd / 'src' / 'mnist_explain_predictions.py').exists():
    SRC_ROOT = cwd / 'src'
elif (cwd.parent / 'mnist_explain_predictions.py').exists():
    SRC_ROOT = cwd.parent
else:
    raise RuntimeError('Could not locate src directory containing mnist_explain_predictions.py')

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from train_mnist_model import MNISTNet
from explainers.lrp import LRP

EXPLANATION_ARTIFACT = SRC_ROOT / 'explanations' / 'mnist' / 'mnist_explanations.pt'
SCORES_CSV = SRC_ROOT / 'explanations' / 'mnist' / 'mnist_explanation_scores.csv'
SUMMARY_CSV = SRC_ROOT / 'explanations' / 'mnist' / 'mnist_method_summary.csv'
EVALUATION_CONFIG_JSON = SRC_ROOT / 'explanations' / 'mnist' / 'mnist_evaluation_config.json'
MODEL_PATH = SRC_ROOT / 'models' / 'mnist' / 'mnist_best_model.pt'
OUTPUT_DIR = SRC_ROOT / 'results' / 'mnist_visualizations' / 'augmentation_case_align'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'SRC_ROOT: {SRC_ROOT}')
print(f'Device: {DEVICE}')
print(f'Output dir: {OUTPUT_DIR}')


In [ ]:
# Experiment parameters.
REFERENCE_METHOD = 'dl'
TARGET_LABELS = [1, 2, 4, 7, 9, 0]
CASE_GROUPS = ['high', 'low']
AUGMENTATIONS_PER_INTENSITY = 10
CASE_ALIGN_K = 10
RANDOM_SEED = 123

# Set this to restrict the run while drafting, for example: [(1, 'low'), (4, 'low'), (9, 'low')]
# Leave as None to run high and low cases for every TARGET_LABELS entry.
SELECTED_CASES = None

# MNIST tensors in the artifacts are normalized with these constants.
MNIST_MEAN = 0.1307
MNIST_STD = 0.3081

# Each variant is one deterministic transformation only. The columns are evenly
# spaced between start and end, so each row reads as a clear visual path from
# the original to a stronger perturbation.
AUGMENTATION_VARIANTS = {
    'rotation': {
        'type': 'rotation',
        'label': 'Rotation',
        'unit': 'deg',
        'levels': {
            'low': {'start': 2.0, 'end': 20.0},
            'mild': {'start': 3.0, 'end': 32.5},
            'intense': {'start': 4.0, 'end': 45.0},
        },
    },
    'shear': {
        'type': 'shear_x',
        'label': 'Horizontal shear',
        'unit': 'deg',
        'levels': {
            'low': {'start': 2.0, 'end': 18.0},
            'mild': {'start': 3.0, 'end': 30.0},
            'intense': {'start': 4.0, 'end': 42.0},
        },
    },
    'scale': {
        'type': 'scale',
        'label': 'Scale up',
        'unit': 'x',
        'levels': {
            'low': {'start': 1.02, 'end': 1.20},
            'mild': {'start': 1.03, 'end': 1.35},
            'intense': {'start': 1.04, 'end': 1.55},
        },
    },
}

CASE_ALIGN_PARAMS = {
    'k': CASE_ALIGN_K,
    'problem_metric': 'cosine',
    'solution_metric': 'cosine',
    'reference_method': REFERENCE_METHOD,
}

EXPERIMENT_PARAMS = {
    'target_labels': TARGET_LABELS,
    'case_groups': CASE_GROUPS,
    'augmentations_per_intensity': AUGMENTATIONS_PER_INTENSITY,
    'augmentation_variants': AUGMENTATION_VARIANTS,
    'selected_cases': SELECTED_CASES,
    'random_seed': RANDOM_SEED,
    'output_dir': str(OUTPUT_DIR),
}

print('Case Align parameters:')
display(pd.Series(CASE_ALIGN_PARAMS, name='value').to_frame())
print('Augmentation experiment parameters:')
display(pd.Series(EXPERIMENT_PARAMS, name='value').to_frame())

if EVALUATION_CONFIG_JSON.exists():
    eval_config = json.loads(EVALUATION_CONFIG_JSON.read_text())
    print('Saved evaluator config:')
    display(pd.Series(eval_config, name='value').to_frame())
    if eval_config.get('sim_metric') != 'cosine':
        print('WARNING: saved baseline Case Align scores were not generated with cosine.')


In [ ]:
artifact = torch.load(EXPLANATION_ARTIFACT, map_location='cpu')
scores_df = pd.read_csv(SCORES_CSV)
summary_df = pd.read_csv(SUMMARY_CSV)

images = artifact['images'].float()  # normalized [N,1,28,28]
labels = artifact['labels'].numpy().astype(int)
pred_labels = artifact['pred_labels'].numpy().astype(int)
confidences = artifact['confidences'].numpy().astype(float)
methods = [m.lower() for m in artifact['methods']]
attributions = {m: artifact['attributions'][m].float() for m in methods}
baseline = artifact.get('baseline', torch.zeros(1, 1, 28, 28)).float().to(DEVICE)

if 'case_align_has_like_neighbour' in scores_df.columns:
    scores_df['case_align_has_like_neighbour'] = scores_df['case_align_has_like_neighbour'].astype(bool)
    scores_df.loc[~scores_df['case_align_has_like_neighbour'], ['case_align_S_plus', 'case_align_R_bounded']] = np.nan

print(f'N images: {len(images)}')
print(f'Methods: {methods}')
display(summary_df.sort_values('method').reset_index(drop=True))

In [ ]:
class MNISTLogitsWrapper(torch.nn.Module):
    def __init__(self, model: torch.nn.Module):
        super().__init__()
        self.model = model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x, return_logits=True)


def load_model(model_path: Path, device: torch.device) -> MNISTNet:
    checkpoint = torch.load(model_path, map_location=device)
    model = MNISTNet()
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    elif isinstance(checkpoint, dict):
        model.load_state_dict(checkpoint)
    else:
        raise RuntimeError('Unsupported checkpoint format')
    model.to(device)
    model.eval()
    return model

model = load_model(MODEL_PATH, DEVICE)
logits_model = MNISTLogitsWrapper(model).to(DEVICE).eval()

if REFERENCE_METHOD == 'ig':
    explainer = IntegratedGradients(model)
elif REFERENCE_METHOD == 'dl':
    explainer = DeepLift(logits_model)
elif REFERENCE_METHOD == 'lrp':
    explainer = LRP(logits_model)
else:
    raise ValueError(f'Unsupported reference method: {REFERENCE_METHOD}')

@torch.no_grad()
def predict_batch(x: torch.Tensor):
    logits = model(x.to(DEVICE))
    probs = torch.softmax(logits, dim=1)
    conf, preds = probs.max(dim=1)
    return preds.detach().cpu().numpy().astype(int), conf.detach().cpu().numpy().astype(float)

def explain_batch(x: torch.Tensor, targets: np.ndarray) -> torch.Tensor:
    x_dev = x.to(DEVICE).clone().detach().requires_grad_(True)
    target_tensor = torch.tensor(targets, dtype=torch.long, device=DEVICE)
    if REFERENCE_METHOD in {'ig', 'dl'}:
        b = baseline.expand_as(x_dev)
        attrs = explainer.attribute(x_dev, baselines=b, target=target_tensor)
    else:
        attrs = explainer.attribute(x_dev, target=target_tensor)
    return attrs.detach().cpu().float()

print(f'Loaded model and {REFERENCE_METHOD} explainer.')

In [ ]:
METHOD_LABEL = {'ig': 'Integrated Gradients', 'dl': 'DeepLift', 'lrp': 'LRP'}

def score_row(method: str, sample_position: int) -> pd.Series:
    row = scores_df[(scores_df['method'] == method) & (scores_df['sample_position'] == sample_position)]
    if row.empty:
        raise ValueError(f'No score row for method={method}, sample_position={sample_position}')
    return row.iloc[0]

def valid_case_align_rows(reference_method: str) -> pd.DataFrame:
    return scores_df[
        (scores_df['method'] == reference_method) &
        scores_df['case_align_S_plus'].notna()
    ][['sample_position', 'true_label', 'case_align_S_plus', 'captum_sensitivity']].copy()

def select_case_align_pair_for_label(reference_method: str, label: int):
    ref = valid_case_align_rows(reference_method)
    selected = ref[ref['true_label'].astype(int) == int(label)].copy()
    if selected.empty:
        raise ValueError(f'No valid Case Align rows for label={label}, method={reference_method}')
    return {
        'high': int(selected.sort_values('case_align_S_plus', ascending=False).iloc[0]['sample_position']),
        'low': int(selected.sort_values('case_align_S_plus', ascending=True).iloc[0]['sample_position']),
        'label': int(label),
        'case_align_min': float(selected['case_align_S_plus'].min()),
        'case_align_max': float(selected['case_align_S_plus'].max()),
        'case_align_spread': float(selected['case_align_S_plus'].max() - selected['case_align_S_plus'].min()),
        'n_valid': int(len(selected)),
    }

pair_selections = {int(label): select_case_align_pair_for_label(REFERENCE_METHOD, int(label)) for label in TARGET_LABELS}

if SELECTED_CASES is None:
    selected_cases = [(int(label), group) for label in TARGET_LABELS for group in CASE_GROUPS]
else:
    selected_cases = [(int(label), str(group)) for label, group in SELECTED_CASES]

anchor_rows = []
for label, group in selected_cases:
    pos = pair_selections[label][group]
    row = score_row(REFERENCE_METHOD, pos)
    anchor_rows.append({
        'label': label,
        'group': group,
        'sample_position': pos,
        'true_label': int(labels[pos]),
        'pred_label': int(pred_labels[pos]),
        'confidence': float(confidences[pos]),
        'case_align_S_plus': float(row['case_align_S_plus']),
        'captum_sensitivity': float(row['captum_sensitivity']),
        'label_case_align_spread': float(pair_selections[label]['case_align_spread']),
    })
anchor_df = pd.DataFrame(anchor_rows)
print('Selected augmentation anchors:')
display(anchor_df)

In [ ]:
def denormalise_mnist(x_norm: torch.Tensor) -> torch.Tensor:
    return torch.clamp(x_norm * MNIST_STD + MNIST_MEAN, 0.0, 1.0)

def normalise_mnist(x_raw: torch.Tensor) -> torch.Tensor:
    return (torch.clamp(x_raw, 0.0, 1.0) - MNIST_MEAN) / MNIST_STD

def normalise_attr(attr_map: np.ndarray, percentile: float = 99.0) -> np.ndarray:
    arr = np.asarray(attr_map, dtype=float).squeeze()
    scale = np.percentile(np.abs(arr), percentile) + 1e-8
    return np.clip(arr / scale, -1.0, 1.0)

def safe_normalise_rows(matrix: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    matrix = np.asarray(matrix, dtype=float)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.maximum(norms, eps)

def cosine_distance_matrix_to_query(matrix: np.ndarray, query_vec: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    matrix_norm = safe_normalise_rows(matrix, eps=eps)
    query_norm = safe_normalise_rows(np.asarray(query_vec, dtype=float)[None, :], eps=eps)[0]
    similarity = matrix_norm @ query_norm
    return np.clip(1.0 - 0.5 * (similarity + 1.0), 0.0, 1.0)

def compute_augmented_case_align(
    query_image: torch.Tensor,
    query_attr: torch.Tensor,
    query_label: int,
    k: int = 10,
    epsilon: float = 1e-8,
):
    pool_images = images.numpy().reshape(len(images), -1).astype(float)
    pool_attrs = attributions[REFERENCE_METHOD].numpy().reshape(len(images), -1).astype(float)
    query_x = query_image.numpy().reshape(-1).astype(float)
    query_e = query_attr.numpy().reshape(-1).astype(float)

    dprob_all = cosine_distance_matrix_to_query(pool_images, query_x)
    like_indices = np.where(labels == int(query_label))[0]
    if like_indices.size == 0:
        return np.nan, 0, [], [], []

    order = np.argsort(dprob_all[like_indices])
    neigh_indices = like_indices[order[:min(k, like_indices.size)]]
    dprob_neigh = dprob_all[neigh_indices]

    dsoln_all = cosine_distance_matrix_to_query(pool_attrs, query_e)
    dsoln_neigh = dsoln_all[neigh_indices]
    ds_min = float(np.min(dsoln_all))
    ds_max = float(np.max(dsoln_all))
    denom = max(ds_max - ds_min, epsilon)
    align = 1.0 - (dsoln_neigh - ds_min) / denom

    weights = 1.0 - dprob_neigh
    weight_sum = float(np.sum(weights))
    if weight_sum <= 0:
        return 0.0, int(len(neigh_indices)), neigh_indices.tolist(), dprob_neigh.tolist(), dsoln_neigh.tolist()

    s_plus = float(np.sum(weights * align) / weight_sum)
    return s_plus, int(len(neigh_indices)), neigh_indices.tolist(), dprob_neigh.tolist(), dsoln_neigh.tolist()

def compute_anchor_case_align_against_generated(
    anchor_image: torch.Tensor,
    anchor_attr: torch.Tensor,
    candidate_images: torch.Tensor,
    candidate_attrs: torch.Tensor,
    candidate_mask: np.ndarray,
    k: int = 10,
    epsilon: float = 1e-8,
):
    mask = np.asarray(candidate_mask, dtype=bool)
    valid_indices = np.where(mask)[0]
    if valid_indices.size == 0:
        return np.nan, 0, [], [], []

    query_x = anchor_image.numpy().reshape(-1).astype(float)
    query_e = anchor_attr.numpy().reshape(-1).astype(float)
    cand_x = candidate_images[valid_indices].numpy().reshape(len(valid_indices), -1).astype(float)
    cand_e = candidate_attrs[valid_indices].numpy().reshape(len(valid_indices), -1).astype(float)

    dprob_candidates = cosine_distance_matrix_to_query(cand_x, query_x)
    order = np.argsort(dprob_candidates)[:min(k, len(valid_indices))]
    neigh_row_indices = valid_indices[order]
    dprob_neigh = dprob_candidates[order]

    dsoln_candidates = cosine_distance_matrix_to_query(cand_e, query_e)
    dsoln_neigh = dsoln_candidates[order]

    # Keep the solution-distance scale comparable to the baseline Case Align calculation.
    pool_attrs = attributions[REFERENCE_METHOD].numpy().reshape(len(images), -1).astype(float)
    dsoln_pool = cosine_distance_matrix_to_query(pool_attrs, query_e)
    ds_min = float(np.min(dsoln_pool))
    ds_max = float(np.max(dsoln_pool))
    denom = max(ds_max - ds_min, epsilon)
    align = np.clip(1.0 - (dsoln_neigh - ds_min) / denom, 0.0, 1.0)

    weights = 1.0 - dprob_neigh
    weight_sum = float(np.sum(weights))
    if weight_sum <= 0:
        return 0.0, int(len(neigh_row_indices)), neigh_row_indices.tolist(), dprob_neigh.tolist(), dsoln_neigh.tolist()

    s_plus = float(np.sum(weights * align) / weight_sum)
    return s_plus, int(len(neigh_row_indices)), neigh_row_indices.tolist(), dprob_neigh.tolist(), dsoln_neigh.tolist()

def transform_values_for_level(params: dict, n_steps: int) -> np.ndarray:
    return np.linspace(float(params['start']), float(params['end']), int(n_steps))

def augment_one(x_norm: torch.Tensor, variant_config: dict, value: float) -> torch.Tensor:
    x_raw = denormalise_mnist(x_norm.detach().cpu()).squeeze(0)  # [1,28,28]
    transform_type = variant_config['type']

    if transform_type == 'rotation':
        aug = TF.rotate(
            x_raw,
            angle=float(value),
            interpolation=InterpolationMode.BILINEAR,
            fill=0.0,
            center=[14, 14],
        )
    elif transform_type == 'shear_x':
        aug = TF.affine(
            x_raw,
            angle=0.0,
            translate=[0, 0],
            scale=1.0,
            shear=[float(value), 0.0],
            interpolation=InterpolationMode.BILINEAR,
            fill=0.0,
            center=[14, 14],
        )
    elif transform_type == 'scale':
        aug = TF.affine(
            x_raw,
            angle=0.0,
            translate=[0, 0],
            scale=float(value),
            shear=[0.0, 0.0],
            interpolation=InterpolationMode.BILINEAR,
            fill=0.0,
            center=[14, 14],
        )
    else:
        raise ValueError(f'Unknown augmentation variant type: {transform_type}')

    return normalise_mnist(aug.unsqueeze(0)).float()  # [1,1,28,28]

def format_transform_value(value: float, unit: str) -> str:
    if unit == 'x':
        return f'{value:.2f}x'
    return f'{value:.1f} {unit}'

print('Deterministic single-transform augmentation and Case Align helpers ready.')


In [ ]:
all_augmented_records = []
anchor_row_records = []
augmented_payload = {}

for variant_name, variant_config in AUGMENTATION_VARIANTS.items():
    augmented_payload[variant_name] = {}
    for label, group in selected_cases:
        anchor_pos = pair_selections[label][group]
        target_label = int(labels[anchor_pos])
        anchor_image = images[anchor_pos:anchor_pos + 1]
        anchor_attr = attributions[REFERENCE_METHOD][anchor_pos:anchor_pos + 1]
        case_key = (label, group)
        augmented_payload[variant_name][case_key] = {}

        for intensity_name, params in variant_config['levels'].items():
            values = transform_values_for_level(params, AUGMENTATIONS_PER_INTENSITY)
            batch = torch.cat([augment_one(anchor_image, variant_config, value) for value in values], dim=0)
            preds, confs = predict_batch(batch)
            same_class_mask = preds.astype(int) == target_label

            if not bool(np.all(same_class_mask)):
                print(
                    f'WARNING: variant={variant_name}, label={label}, group={group}, intensity={intensity_name}: '
                    f'{int(np.sum(same_class_mask))}/{AUGMENTATIONS_PER_INTENSITY} generated samples retained the target class {target_label}.'
                )

            attrs = torch.empty(batch.shape[0], 1, 28, 28)
            if bool(np.any(same_class_mask)):
                same_idx = np.where(same_class_mask)[0]
                same_batch = batch[same_idx]
                same_preds = preds[same_idx]
                attrs[same_idx] = explain_batch(same_batch, same_preds)

            anchor_s_plus, anchor_n, anchor_neigh_idx, anchor_dprob, anchor_dsoln = compute_anchor_case_align_against_generated(
                anchor_image,
                anchor_attr,
                batch,
                attrs,
                same_class_mask,
                k=CASE_ALIGN_K,
            )

            augmented_payload[variant_name][case_key][intensity_name] = {
                'images': batch,
                'preds': preds,
                'confidences': confs,
                'attributions': attrs,
                'same_class': same_class_mask,
                'values': values,
                'anchor_case_align_S_plus': anchor_s_plus,
                'anchor_n_generated_neighbours': anchor_n,
                'anchor_generated_neighbour_indices': anchor_neigh_idx,
                'anchor_generated_neighbour_problem_distances': anchor_dprob,
                'anchor_generated_neighbour_solution_distances': anchor_dsoln,
            }

            anchor_row_records.append({
                'augmentation_variant': variant_name,
                'augmentation_label': variant_config['label'],
                'augmentation_unit': variant_config['unit'],
                'label': label,
                'group': group,
                'anchor_position': anchor_pos,
                'intensity': intensity_name,
                'anchor_case_align_S_plus': float(anchor_s_plus) if np.isfinite(anchor_s_plus) else np.nan,
                'anchor_n_generated_neighbours': int(anchor_n),
                'anchor_generated_neighbour_indices': anchor_neigh_idx,
                'anchor_mean_problem_distance_to_generated_neighbours': float(np.mean(anchor_dprob)) if len(anchor_dprob) else np.nan,
                'anchor_mean_solution_distance_to_generated_neighbours': float(np.mean(anchor_dsoln)) if len(anchor_dsoln) else np.nan,
            })

            for i in range(batch.shape[0]):
                if same_class_mask[i]:
                    s_plus, n_like, neigh_idx, dprob, dsoln = compute_augmented_case_align(
                        batch[i:i + 1], attrs[i:i + 1], target_label, k=CASE_ALIGN_K
                    )
                else:
                    s_plus, n_like, neigh_idx, dprob, dsoln = np.nan, 0, [], [], []

                all_augmented_records.append({
                    'augmentation_variant': variant_name,
                    'augmentation_label': variant_config['label'],
                    'augmentation_unit': variant_config['unit'],
                    'label': label,
                    'group': group,
                    'anchor_position': anchor_pos,
                    'intensity': intensity_name,
                    'transform_value': float(values[i]),
                    'augmentation_index': i,
                    'pred_label': int(preds[i]),
                    'confidence': float(confs[i]),
                    'same_class': bool(same_class_mask[i]),
                    'case_align_S_plus': float(s_plus) if np.isfinite(s_plus) else np.nan,
                    'n_like': int(n_like),
                    'mean_problem_distance_to_neighbours': float(np.mean(dprob)) if len(dprob) else np.nan,
                    'mean_solution_distance_to_neighbours': float(np.mean(dsoln)) if len(dsoln) else np.nan,
                })

augmented_df = pd.DataFrame(all_augmented_records)
anchor_row_df = pd.DataFrame(anchor_row_records)
summary_aug_df = augmented_df.groupby(['augmentation_variant', 'augmentation_label', 'label', 'group', 'intensity'], as_index=False).agg(
    n=('case_align_S_plus', 'size'),
    n_same_class=('same_class', 'sum'),
    mean_case_align=('case_align_S_plus', 'mean'),
    std_case_align=('case_align_S_plus', 'std'),
    mean_confidence=('confidence', 'mean'),
    mean_problem_distance=('mean_problem_distance_to_neighbours', 'mean'),
    mean_solution_distance=('mean_solution_distance_to_neighbours', 'mean'),
).merge(
    anchor_row_df[[
        'augmentation_variant',
        'label',
        'group',
        'intensity',
        'anchor_case_align_S_plus',
        'anchor_n_generated_neighbours',
        'anchor_mean_problem_distance_to_generated_neighbours',
        'anchor_mean_solution_distance_to_generated_neighbours',
    ]],
    on=['augmentation_variant', 'label', 'group', 'intensity'],
    how='left',
)

csv_path = OUTPUT_DIR / 'mnist_augmentation_case_align_records.csv'
summary_csv_path = OUTPUT_DIR / 'mnist_augmentation_case_align_summary.csv'
anchor_csv_path = OUTPUT_DIR / 'mnist_augmentation_anchor_generated_neighbour_scores.csv'
augmented_df.to_csv(csv_path, index=False)
summary_aug_df.to_csv(summary_csv_path, index=False)
anchor_row_df.to_csv(anchor_csv_path, index=False)
print(f'Saved records: {csv_path}')
print(f'Saved summary: {summary_csv_path}')
print(f'Saved anchor generated-neighbour scores: {anchor_csv_path}')
print('Augmentation Case Align summary:')
display(summary_aug_df)
print('Original anchor scored against generated neighbours per row:')
display(anchor_row_df)


In [ ]:
AUG_FIG_DPI = 300
AUG_TITLE_FS = 9
AUG_SUPTITLE_FS = 13

plot_paths = []

for variant_name, variant_config in AUGMENTATION_VARIANTS.items():
    intensity_order = list(variant_config['levels'].keys())
    unit = variant_config['unit']
    variant_label = variant_config['label']

    for label, group in selected_cases:
        anchor_pos = pair_selections[label][group]
        baseline_anchor_score = score_row(REFERENCE_METHOD, anchor_pos)
        case_key = (label, group)
        fig, axes = plt.subplots(len(intensity_order), AUGMENTATIONS_PER_INTENSITY + 1, figsize=(2.15 * (AUGMENTATIONS_PER_INTENSITY + 1), 6.8))
        fig.subplots_adjust(wspace=0.05, hspace=0.38, right=0.93, left=0.04, top=0.84, bottom=0.05)
        if axes.ndim == 1:
            axes = axes[np.newaxis, :]

        im = None
        for row_idx, intensity_name in enumerate(intensity_order):
            payload = augmented_payload[variant_name][case_key][intensity_name]
            batch = payload['images']
            attrs = payload['attributions']
            preds = payload['preds']
            confs = payload['confidences']
            same_class = payload['same_class']
            values = payload['values']
            intensity_records = augmented_df[
                (augmented_df['augmentation_variant'] == variant_name) &
                (augmented_df['label'] == label) &
                (augmented_df['group'] == group) &
                (augmented_df['intensity'] == intensity_name)
            ].sort_values('augmentation_index')

            ax0 = axes[row_idx, 0]
            ax0.imshow(denormalise_mnist(images[anchor_pos:anchor_pos + 1])[0, 0], cmap='gray', interpolation='bicubic')
            row_anchor_s_plus = payload['anchor_case_align_S_plus']
            row_anchor_n = payload['anchor_n_generated_neighbours']
            if np.isfinite(row_anchor_s_plus):
                anchor_title = (
                    f'{intensity_name.title()}\nOriginal vs row NNs\n'
                    f'S+={row_anchor_s_plus:.3f}, k={row_anchor_n}'
                )
            else:
                anchor_title = f'{intensity_name.title()}\nOriginal vs row NNs\nno same-class NNs'
            ax0.set_title(anchor_title, fontsize=AUG_TITLE_FS)
            ax0.axis('off')
            ax0.text(
                -0.12, 0.5, intensity_name.title(), transform=ax0.transAxes,
                rotation=90, va='center', ha='right', fontsize=11, fontweight='bold'
            )

            for col in range(AUGMENTATIONS_PER_INTENSITY):
                ax = axes[row_idx, col + 1]
                ax.imshow(denormalise_mnist(batch[col:col + 1])[0, 0], cmap='gray', interpolation='bicubic')
                record = intensity_records[intensity_records['augmentation_index'] == col].iloc[0]
                value_label = format_transform_value(float(values[col]), unit)

                if same_class[col]:
                    attr = normalise_attr(attrs[col, 0].numpy())
                    im = ax.imshow(attr, cmap='RdBu_r', vmin=-1, vmax=1, alpha=0.58, interpolation='bicubic')
                    title = (
                        f'{value_label}\nS+={record["case_align_S_plus"]:.3f}\n'
                        f'pred={int(preds[col])}, conf={confs[col]:.2f}'
                    )
                else:
                    title = (
                        f'{value_label}\npred changed\n'
                        f'pred={int(preds[col])}, conf={confs[col]:.2f}'
                    )
                ax.set_title(title, fontsize=AUG_TITLE_FS)
                ax.axis('off')

        if im is not None:
            cbar_ax = fig.add_axes([0.945, 0.18, 0.010, 0.60])
            cbar = fig.colorbar(im, cax=cbar_ax)
            cbar.set_label('Attribution overlay', rotation=270, labelpad=12)

        fig.suptitle(
            f'{METHOD_LABEL.get(REFERENCE_METHOD, REFERENCE_METHOD.upper())}: label {label} {group} {variant_label.lower()} degradation; original scored against row generated neighbours (baseline S+={baseline_anchor_score["case_align_S_plus"]:.3f})',
            fontsize=AUG_SUPTITLE_FS,
        )

        png_path = OUTPUT_DIR / f'mnist_label_{label}_{group}_{variant_name}_augmentation_case_align_degradation.png'
        pdf_path = OUTPUT_DIR / f'mnist_label_{label}_{group}_{variant_name}_augmentation_case_align_degradation.pdf'
        fig.savefig(png_path, dpi=AUG_FIG_DPI, bbox_inches='tight')
        fig.savefig(pdf_path, dpi=AUG_FIG_DPI, bbox_inches='tight')
        plot_paths.append((variant_name, png_path, pdf_path))
        print(f'Saved: {png_path}')
        print(f'Saved: {pdf_path}')
        plt.close(fig)

print('Augmentation degradation figures:')
for variant_name, png_path, pdf_path in plot_paths:
    print(variant_name, png_path)
    print(variant_name, pdf_path)
